# Fast PCB Component Classification — Kaggle
MobileNetV3-Small compact head, cached baseline and short partial fine-tuning, plus EfficientNet-B0 challenger. Enable GPU and Internet; attach GraphPCB. No extra PCB images. Read the embedded README for definitions, annotation limitations and resume instructions. Full-dataset accuracy and training time have not been measured.

In [ ]:
from pathlib import Path
import sys,os,json,subprocess,shutil
SOURCE = '' # EDIT: /kaggle/input/.../GraphPCB or .../archive.zip
PROJECT=Path('/kaggle/working/Classification')
DATA=Path('/kaggle/temp/classification_prepared')
OUT=Path('/kaggle/working/results/classification')
RESUME_FROM='' # optional previous saved classification output directory
if RESUME_FROM and not OUT.exists(): shutil.copytree(RESUME_FROM,OUT)
PROJECT.mkdir(parents=True,exist_ok=True)
subprocess.run(['nvidia-smi'],check=True)
subprocess.run([sys.executable,'-m','pip','install','pycocotools==2.0.11'],check=True)

In [ ]:
SOURCES = {'config.yml': 'seed: 42\nclasses: 25\nbatch: 128\npatience: 3\nextra_pcb_images: 0\ntrials: [small_cached160, small_tail192, b0_tail192]\n', 'data_mapping.json': '{\n  "canonical_classes": [\n    {\n      "id": 1,\n      "code": "R",\n      "source_rgb": [\n        255,\n        0,\n        0\n      ],\n      "display_hex": "#E53935"\n    },\n    {\n      "id": 2,\n      "code": "C",\n      "source_rgb": [\n        255,\n        255,\n        0\n      ],\n      "display_hex": "#1E88E5"\n    },\n    {\n      "id": 3,\n      "code": "U",\n      "source_rgb": [\n        0,\n        234,\n        255\n      ],\n      "display_hex": "#00ACC1"\n    },\n    {\n      "id": 4,\n      "code": "Q",\n      "source_rgb": [\n        170,\n        0,\n        255\n      ],\n      "display_hex": "#8E24AA"\n    },\n    {\n      "id": 5,\n      "code": "J",\n      "source_rgb": [\n        255,\n        127,\n        0\n      ],\n      "display_hex": "#FB8C00"\n    },\n    {\n      "id": 6,\n      "code": "L",\n      "source_rgb": [\n        191,\n        255,\n        0\n      ],\n      "display_hex": "#43A047"\n    },\n    {\n      "id": 7,\n      "code": "RA",\n      "source_rgb": [\n        0,\n        149,\n        255\n      ],\n      "display_hex": "#6D4C41"\n    },\n    {\n      "id": 8,\n      "code": "D",\n      "source_rgb": [\n        106,\n        255,\n        0\n      ],\n      "display_hex": "#D81B60"\n    },\n    {\n      "id": 9,\n      "code": "RN",\n      "source_rgb": [\n        0,\n        64,\n        255\n      ],\n      "display_hex": "#3949AB"\n    },\n    {\n      "id": 10,\n      "code": "TP",\n      "source_rgb": [\n        237,\n        185,\n        185\n      ],\n      "display_hex": "#C0CA33"\n    },\n    {\n      "id": 11,\n      "code": "IC",\n      "source_rgb": [\n        185,\n        215,\n        237\n      ],\n      "display_hex": "#00897B"\n    },\n    {\n      "id": 12,\n      "code": "P",\n      "source_rgb": [\n        231,\n        233,\n        185\n      ],\n      "display_hex": "#FFB300"\n    },\n    {\n      "id": 13,\n      "code": "CR",\n      "source_rgb": [\n        220,\n        185,\n        237\n      ],\n      "display_hex": "#5E35B1"\n    },\n    {\n      "id": 14,\n      "code": "M",\n      "source_rgb": [\n        185,\n        237,\n        224\n      ],\n      "display_hex": "#039BE5"\n    },\n    {\n      "id": 15,\n      "code": "BTN",\n      "source_rgb": [\n        143,\n        35,\n        35\n      ],\n      "display_hex": "#757575"\n    },\n    {\n      "id": 16,\n      "code": "FB",\n      "source_rgb": [\n        35,\n        98,\n        143\n      ],\n      "display_hex": "#FDD835"\n    },\n    {\n      "id": 17,\n      "code": "CRA",\n      "source_rgb": [\n        143,\n        106,\n        35\n      ],\n      "display_hex": "#212121"\n    },\n    {\n      "id": 18,\n      "code": "SW",\n      "source_rgb": [\n        107,\n        35,\n        143\n      ],\n      "display_hex": "#7E57C2"\n    },\n    {\n      "id": 19,\n      "code": "T",\n      "source_rgb": [\n        79,\n        143,\n        35\n      ],\n      "display_hex": "#F4511E"\n    },\n    {\n      "id": 20,\n      "code": "F",\n      "source_rgb": [\n        115,\n        115,\n        115\n      ],\n      "display_hex": "#2E7D32"\n    },\n    {\n      "id": 21,\n      "code": "V",\n      "source_rgb": [\n        204,\n        204,\n        204\n      ],\n      "display_hex": "#283593"\n    },\n    {\n      "id": 22,\n      "code": "LED",\n      "source_rgb": [\n        245,\n        130,\n        48\n      ],\n      "display_hex": "#7CB342"\n    },\n    {\n      "id": 23,\n      "code": "S",\n      "source_rgb": [\n        220,\n        190,\n        255\n      ],\n      "display_hex": "#546E7A"\n    },\n    {\n      "id": 24,\n      "code": "QA",\n      "source_rgb": [\n        170,\n        255,\n        195\n      ],\n      "display_hex": "#C2185B"\n    },\n    {\n      "id": 25,\n      "code": "JP",\n      "source_rgb": [\n        255,\n        250,\n        200\n      ],\n      "display_hex": "#F9A825"\n    }\n  ],\n  "source_palette": "https://github.com/CandleLabAI/PCBSegClassNet/blob/main/src/data/create_mask.py",\n  "graph_w_type_mapping": {\n    "resistor": "R",\n    "capacitor": "C",\n    "electrolytic capacitor": "C",\n    "ic": "IC",\n    "transistor": "Q",\n    "connector": "J",\n    "inductor": "L",\n    "diode": "D",\n    "zener diode": "D",\n    "diode zener array": "D",\n    "resistor network": "RN",\n    "test point": "TP",\n    "button": "BTN",\n    "ferrite bead": "FB",\n    "switch": "SW",\n    "fuse": "F",\n    "led": "LED",\n    "jumper": "JP"\n  },\n  "graph_f_instance_rule": "8-connected regions of the exact RGB class; touching same-class objects may merge and fragments may split",\n  "graph_w_mask_rule": "Loader rectangles only; Graph-W mask loss excluded; Graph-W true mask metrics unavailable",\n  "unmapped_source_types": "ignore/void; source type counts are generated from official training XML only during preparation",\n  "external_pcb_images": 0\n}', 'inference_classification.py': '"""Standalone 25-class model construction, preprocessing, metrics and box overlays."""\nimport argparse,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom PIL import Image,ImageOps,ImageDraw\nfrom torchvision import models,transforms as T\nCLASSES=(\'R\',\'C\',\'U\',\'Q\',\'J\',\'L\',\'RA\',\'D\',\'RN\',\'TP\',\'IC\',\'P\',\'CR\',\'M\',\'BTN\',\'FB\',\'CRA\',\'SW\',\'T\',\'F\',\'V\',\'LED\',\'S\',\'QA\',\'JP\')\nCOLORS=[\'#E53935\',\'#1E88E5\',\'#00ACC1\',\'#8E24AA\',\'#FB8C00\',\'#43A047\',\'#6D4C41\',\'#D81B60\',\'#3949AB\',\'#C0CA33\',\'#00897B\',\'#FFB300\',\'#5E35B1\',\'#039BE5\',\'#757575\',\'#FDD835\',\'#212121\',\'#7E57C2\',\'#F4511E\',\'#2E7D32\',\'#283593\',\'#7CB342\',\'#546E7A\',\'#C2185B\',\'#F9A825\']\ndef save(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);path.write_text(json.dumps(obj,indent=2,allow_nan=False))\ndef build(arch,tail=0,pretrained=False):\n    if arch==\'small\':\n        m=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None);width=576\n    else:\n        m=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None);width=1280\n    m.classifier=nn.Linear(width,25)\n    for p in m.features.parameters():p.requires_grad=False\n    if tail:\n        for block in list(m.features.children())[-tail:]:\n            for p in block.parameters():p.requires_grad=True\n    return m\nclass Square:\n    def __init__(self,size):self.size=size\n    def __call__(self,im):return ImageOps.pad(im,(self.size,self.size),method=Image.Resampling.BILINEAR,color=(114,114,114))\ndef preprocessing(size,augment=False):\n    steps=[Square(size)]\n    if augment:steps += [T.RandomHorizontalFlip(),T.RandomVerticalFlip(),T.RandomRotation(15,fill=114),T.ColorJitter(.15,.15,.1,.02)]\n    return T.Compose(steps+[T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])\ndef metrics(y,prob,t):\n    y=np.asarray(y);p=prob.argmax(1);accepted=prob.max(1)>=t\n    tp=np.bincount(y[(p==y)&accepted],minlength=25)\n    predicted=np.bincount(p[accepted],minlength=25);support=np.bincount(y,minlength=25)\n    precision=np.divide(tp,predicted,out=np.zeros(25),where=predicted>0)\n    recall=np.divide(tp,support,out=np.zeros(25),where=support>0)\n    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(25),where=precision+recall>0)\n    return dict(P_macro=float(precision.mean()),R_macro=float(recall.mean()),F1_macro=float(f1.mean()),\n        S_cls=float(.2*precision.mean()+.2*recall.mean()+.6*f1.mean()),threshold=float(t),\n        accuracy=float(((p==y)&accepted).mean()),coverage=float(accepted.mean()),\n        per_class=[dict(code=c,precision=float(precision[i]),recall=float(recall[i]),F1=float(f1[i]),support=int(support[i])) for i,c in enumerate(CLASSES)])\ndef curves(y,p):return [metrics(y,p,float(t)) for t in np.linspace(0,1,101)]\ndef load(path,device):\n    c=torch.load(path,map_location=\'cpu\',weights_only=False);m=build(c[\'arch\'],c[\'tail\']);m.load_state_dict(c[\'model\']);return m.to(device).eval(),c\ndef crop_box(image,box):\n    x,y,w,h=box;margin=.1*max(w,h)\n    return image.crop((max(0,int(x-margin)),max(0,int(y-margin)),min(image.width,int(np.ceil(x+w+margin))),min(image.height,int(np.ceil(y+h+margin)))))\ndef boards(model,meta,annotation,root,out,device):\n    co=json.loads(Path(annotation).read_text());by={}\n    for a in co[\'annotations\']:by.setdefault(a[\'image_id\'],[]).append(a)\n    tf=preprocessing(meta[\'size\']);rows=[]\n    for im in co[\'images\']:\n        image=Image.open(Path(root)/im[\'file_name\']).convert(\'RGB\');canvas=image.copy();anns=by.get(im[\'id\'],[]);draw=ImageDraw.Draw(canvas)\n        for start in range(0,len(anns),128):\n            chunk=anns[start:start+128]\n            batch=torch.stack([tf(crop_box(image,a[\'bbox\'])) for a in chunk]).to(device)\n            with torch.inference_mode():probs=model(batch).softmax(1).cpu().numpy()\n            for a,p in zip(chunk,probs):\n                label=int(p.argmax());confidence=float(p.max());accepted=confidence>=meta[\'threshold\']\n                rows.append(dict(image_id=im[\'id\'],true_class=a[\'category_id\']-1,predicted_class=label,confidence=confidence,accepted=accepted))\n                x,y,w,h=a[\'bbox\'];draw.rectangle((x,y,x+w,y+h),outline=COLORS[label] if accepted else \'#FFFFFF\',width=3)\n        dest=Path(out)/f"{im[\'id\']:04d}.png";dest.parent.mkdir(parents=True,exist_ok=True);canvas.thumbnail((1800,1800));canvas.save(dest)\n    save(Path(out)/\'predictions.json\',rows)\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--checkpoint\',required=True);p.add_argument(\'--image\');p.add_argument(\'--coco\');p.add_argument(\'--root\',default=\'.\');p.add_argument(\'--out\',default=\'predictions\');a=p.parse_args()\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\';m,c=load(a.checkpoint,device)\n    if a.coco:boards(m,c,a.coco,a.root,a.out,device)\n    elif a.image:\n        with torch.inference_mode():prob=m(preprocessing(c[\'size\'])(Image.open(a.image).convert(\'RGB\')).unsqueeze(0).to(device)).softmax(1)[0]\n        print(dict(code=CLASSES[int(prob.argmax())],confidence=float(prob.max()),accepted=bool(prob.max()>=c[\'threshold\'])))\n    else:p.error(\'Supply a component crop --image or annotated boards --coco and --root\')\n', 'README.md': '# Fast PCB component classification on Kaggle\n\nRecommendation: MobileNetV3-Small with global pooled features and a direct 25-class linear head. Removing its original large classifier reduces parameters. EfficientNet-B0 is a validation challenger, not an assumed winner.\n\nImport PCB_Classification_Kaggle.ipynb, add GraphPCB as input, enable GPU + Internet, set SOURCE to the GraphPCB folder or archive.zip, and run all cells. All source is embedded. Prepared crops are in /kaggle/temp; save Kaggle notebook outputs to retain checkpoints. Sessions still have limits. To resume in a new session, attach the previous saved outputs and copy the previous results/classification directory to the same output path before training. Do not change model settings when resuming.\n\nThree trials: small_cached160 caches frozen ImageNet features once, then trains only a linear head for at most 20 cheap epochs; small_tail192 fine-tunes the last three feature blocks for at most 8 epochs; b0_tail192 fine-tunes the last two blocks for at most 8 epochs. All stop after three validation checks without improvement. Fixed early layers and their BN statistics remain frozen throughout each trial. Total and trainable counts are both disclosed. No claimed runtime or accuracy is available until the real run. Resume is epoch-level, not bitwise reproducibility after interruption; cached features are recomputed on restart.\n\nSpeed choices: extract crops once at up to 256 pixels, use 160/192 inputs, batch 128, mixed precision, cached frozen features, and partial fine-tuning. If out of memory use batch 64 in a new experiment directory. No training examples are subsampled. Cached baseline has no image augmentation; fine-tuned trials use flips, small rotations and modest color jitter. Square padding preserves aspect ratio and 10% box context. Trials compare training recipes, not a single isolated architectural factor.\n\nGraph-W XML component boxes provide classification targets. Graph-F instances are 8-connected exact-color regions; touching same-class objects and separated fragments remain an annotation limitation. Graph files are used only for official membership, never unpickled or used as four-class labels. Source taxonomy/mapping is shared with the segmentation code. No external PCB images are added. ImageNet pretraining is disclosed.\n\nTraining-only class-preserving board groups determine validation before crop extraction. Official test membership remains 47 Graph-F + 10 Graph-W boards. Validation checkpoint and confidence threshold maximize S_cls = .2 P_macro + .2 R_macro + .6 F1_macro. All macro averages include exactly 25 classes, undefined per-class scores zero. A rejected crop is still a false negative for its true class; it is never removed from the evaluation denominator. Incorrect accepted predictions contribute both FP and FN. Curves sweep [0,1]. The selected trial and thresholds are frozen before test evaluation. Test scores must not be used to retune.\n\nOutputs: classification_stats.json (consistent trial fields), per-class metrics, P/R/F1 threshold curves, train/val losses, validation accuracy, confusion matrices with an explicit rejection column, predicted-color boxes on every test board, all trial checkpoints and best_classification.ckpt. Annotation boxes are supplied at board inference: this is component classification, not box detection. Standalone inference: python inference_classification.py --checkpoint results/classification/best_classification.ckpt --image component_crop.jpg; or --coco prepared/test_boards.json --root prepared --out predictions. Only external packages are imported by that file. scripts/run_classification.sh invokes it.\n\nKeep the best validation-selected trial. The cohort median parameter size is unknown, so exact efficiency utility/marks cannot be guaranteed. Freezing parameters is disclosed, not described as a smaller stored model. The cached baseline may be very fast but may lose accuracy; use validation results to decide. Both total parameters and predictive quality matter. These short defaults prioritize turnaround rather than claiming an accuracy-preserving shortcut. Extend only the promising trial if validation is still improving, in a new clearly logged experiment; do not tune from test results.\n\nSources: https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_small.html ; https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html ; https://github.com/BlueSkyLT/GraphPCB ; https://github.com/CandleLabAI/PCBSegClassNet/blob/main/src/data/create_mask.py . Torchvision layers and ImageNet weights are third-party resources. Group-specific repository, combined archive and submission organization remain your responsibility.\n', 'TESTING.md': 'Notebook schema and every Python/code-cell syntax were checked. Actual randomly initialized MobileNetV3-Small and EfficientNet-B0 25-class forward passes and parameter counts were checked on CPU, together with confidence-rejection metrics. Full GraphPCB classification training, cached-feature optimization, GPU AMP and Kaggle execution have not been benchmarked. No accuracy or runtime result is claimed. The dataset parser/mapping/split are reused from the previously audited segmentation converter.\n', 'scripts/run_classification.sh': '#!/usr/bin/env bash\nset -euo pipefail\nHERE="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")/.." && pwd)"\npython "$HERE/inference_classification.py" "$@"\n', 'src/classification/prepare.py': '"""Extract component crops directly; no segmentation tile generation is needed."""\nimport sys,json,io,hashlib,argparse,collections\nfrom pathlib import Path\nfrom PIL import Image\nimport numpy as np\nfrom source_data import open_source,inventory,training_class_presence,make_split,f_instances,w_instances,coco_base\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom inference_classification import crop_box,save\ndef prepare(source,out):\n    out=Path(out)\n    if (out/\'manifest.json\').exists():raise FileExistsError(\'Use the existing prepared dataset or a new output path\')\n    z=open_source(source);rs=inventory(z);rs=make_split(rs,42,class_presence=training_class_presence(z,rs));rows=[];co={s:coco_base() for s in [\'train\',\'val\',\'test\']};seen={}\n    for iid,r in enumerate(rs,1):\n        raw=z.read(r[\'zip_image\']);sha=hashlib.sha256(raw).hexdigest()\n        if sha in seen and seen[sha]!=r[\'official_split\']:raise ValueError(\'Exact duplicate across official train/test\')\n        seen[sha]=r[\'official_split\'];im=Image.open(io.BytesIO(raw)).convert(\'RGB\');w,h=im.size\n        objs=f_instances(np.array(Image.open(io.BytesIO(z.read(r[\'zip_annotation\']))).convert(\'RGB\'))) if r[\'domain\']==\'Graph-F\' else w_instances(z.read(r[\'zip_annotation\']),w,h)[0]\n        rel=Path(\'boards\')/r[\'domain\']/Path(r[\'zip_image\']).name;(out/rel).parent.mkdir(parents=True,exist_ok=True);(out/rel).write_bytes(raw)\n        co[r[\'split\']][\'images\'].append(dict(id=iid,file_name=rel.as_posix(),width=w,height=h))\n        for j,(cid,x,y,bw,bh,*_) in enumerate(objs):\n            box=[x,y,bw,bh];name=Path(\'crops\')/r[\'split\']/f\'{iid}_{j}.jpg\';(out/name).parent.mkdir(parents=True,exist_ok=True)\n            crop=crop_box(im,box);crop.thumbnail((256,256));crop.save(out/name,quality=95)\n            rows.append(dict(file=name.as_posix(),label=cid-1,split=r[\'split\'],board_id=iid))\n            co[r[\'split\']][\'annotations\'].append(dict(id=len(co[r[\'split\']][\'annotations\'])+1,image_id=iid,category_id=cid,bbox=box))\n        print(iid,len(rs),r[\'stem\'],len(objs),flush=True)\n    for s,c in co.items():save(out/f\'{s}_boards.json\',c)\n    save(out/\'crops.json\',rows);save(out/\'manifest.json\',dict(records=rs,seed=42,data_ratio=sum(r[\'split\']==\'train\' for r in rs)/152,extra_images=0))\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--source\',required=True);p.add_argument(\'--out\',required=True);a=p.parse_args();prepare(a.source,a.out)\n', 'src/classification/source_data.py': '"""Convert the supplied GraphPCB ZIP without unpickling graph files.\n\nGraph-F instances are recovered from connected regions in the semantic masks.\nGraph-W has boxes only: its rectangles are API placeholders, never mask targets.\n"""\nimport argparse\nimport collections\nimport hashlib\nimport io\nimport json\nimport re\nimport shlex\nimport zipfile\nfrom types import SimpleNamespace\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\nfrom pycocotools import mask as mu\n\nCLASSES = (\'R\',\'C\',\'U\',\'Q\',\'J\',\'L\',\'RA\',\'D\',\'RN\',\'TP\',\'IC\',\'P\',\'CR\',\'M\',\n           \'BTN\',\'FB\',\'CRA\',\'SW\',\'T\',\'F\',\'V\',\'LED\',\'S\',\'QA\',\'JP\')\nSOURCE_RGB = [(255,0,0),(255,255,0),(0,234,255),(170,0,255),(255,127,0),\n              (191,255,0),(0,149,255),(106,255,0),(0,64,255),(237,185,185),\n              (185,215,237),(231,233,185),(220,185,237),(185,237,224),\n              (143,35,35),(35,98,143),(143,106,35),(107,35,143),(79,143,35),\n              (115,115,115),(204,204,204),(245,130,48),(220,190,255),\n              (170,255,195),(255,250,200)]\nDISPLAY_HEX = [\'#E53935\',\'#1E88E5\',\'#00ACC1\',\'#8E24AA\',\'#FB8C00\',\'#43A047\',\n               \'#6D4C41\',\'#D81B60\',\'#3949AB\',\'#C0CA33\',\'#00897B\',\'#FFB300\',\n               \'#5E35B1\',\'#039BE5\',\'#757575\',\'#FDD835\',\'#212121\',\'#7E57C2\',\n               \'#F4511E\',\'#2E7D32\',\'#283593\',\'#7CB342\',\'#546E7A\',\'#C2185B\',\n               \'#F9A825\']\n# Exact source type, parsed with shlex to preserve quoted multiword names.\nW_MAP = {\'resistor\':\'R\', \'capacitor\':\'C\', \'electrolytic capacitor\':\'C\',\n         \'ic\':\'IC\', \'transistor\':\'Q\', \'connector\':\'J\', \'inductor\':\'L\',\n         \'diode\':\'D\', \'zener diode\':\'D\', \'diode zener array\':\'D\',\n         \'resistor network\':\'RN\', \'test point\':\'TP\', \'button\':\'BTN\',\n         \'ferrite bead\':\'FB\', \'switch\':\'SW\', \'fuse\':\'F\', \'led\':\'LED\',\n         \'jumper\':\'JP\'}\nPALETTE_SOURCE = \'https://github.com/CandleLabAI/PCBSegClassNet/blob/main/src/data/create_mask.py\'\n\n\nclass DirectoryArchive:\n    """Read a Kaggle-expanded GraphPCB directory through the ZIP reader interface."""\n    def __init__(self,root):\n        root=Path(root)\n        self.files={(\'GraphPCB/\'+p.relative_to(root).as_posix()):p\n                    for domain in (\'Graph-F\',\'Graph-W\') for p in sorted((root/domain).rglob(\'*\'))\n                    if p.is_file()}\n    def namelist(self): return list(self.files)\n    def read(self,name): return self.files[name].read_bytes()\n    def infolist(self):\n        return [SimpleNamespace(filename=n,CRC=0,file_size=p.stat().st_size)\n                for n,p in self.files.items()]\n\n\ndef open_source(source):\n    return DirectoryArchive(source) if Path(source).is_dir() else zipfile.ZipFile(source)\n\n\ndef dump(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding=\'utf8\')\n\n\ndef json_rle(rle):\n    return {\'size\': list(rle[\'size\']), \'counts\': rle[\'counts\'].decode(\'ascii\')\n            if isinstance(rle[\'counts\'], bytes) else rle[\'counts\']}\n\n\ndef region_rle(crop, x, y, height, width):\n    """Place a small binary crop in a full-size RLE, without allocating H*W."""\n    yy, xx = np.nonzero(crop.T)  # transpose: first index is the original x\n    if not len(xx):\n        return json_rle(mu.frPyObjects({\'size\':[height,width],\n                                        \'counts\':[height*width]}, height, width))\n    positions = (yy.astype(np.int64) + x) * height + xx + y\n    breaks = np.flatnonzero(np.diff(positions) != 1)\n    starts = positions[np.r_[0, breaks + 1]]\n    ends = positions[np.r_[breaks, len(positions)-1]]\n    counts = [int(starts[0])]\n    for i, (start, end) in enumerate(zip(starts, ends)):\n        if i:\n            counts.append(int(start - ends[i-1] - 1))\n        counts.append(int(end - start + 1))\n    counts.append(int(height*width - ends[-1] - 1))\n    return json_rle(mu.frPyObjects({\'size\':[height,width], \'counts\':counts},\n                                   height, width))\n\n\ndef positions(length, tile, overlap):\n    if length <= tile:\n        return [0]\n    starts = list(range(0, length-tile+1, tile-overlap))\n    if starts[-1] != length-tile:\n        starts.append(length-tile)\n    return starts\n\n\ndef f_instances(rgb):\n    """Keep every connected component, including single-pixel regions."""\n    packed = (rgb[:,:,0].astype(np.uint32)<<16) | \\\n             (rgb[:,:,1].astype(np.uint32)<<8) | rgb[:,:,2].astype(np.uint32)\n    lut = np.full(1<<24, 255, np.uint8)\n    lut[0] = 0\n    for k, (r,g,b) in enumerate(SOURCE_RGB, 1):\n        lut[(r<<16)|(g<<8)|b] = k\n    semantic = lut[packed]\n    if np.any(semantic == 255):\n        unknown = np.unique(packed[semantic == 255])\n        raise ValueError(f\'Unknown Graph-F mask colors: {unknown[:20].tolist()}. \'\n                         \'Do not guess a nearest color or a class mapping.\')\n    objects = []\n    for cid in np.unique(semantic):\n        if cid == 0:\n            continue\n        n, labels, stats, _ = cv2.connectedComponentsWithStats(\n            (semantic == cid).astype(np.uint8), connectivity=8)\n        for instance_id in range(1, n):\n            x,y,w,h,area = map(int, stats[instance_id])\n            crop = (labels[y:y+h,x:x+w] == instance_id).astype(np.uint8)\n            objects.append((int(cid), x,y,w,h,crop, int(area)))\n    return objects\n\n\ndef w_instances(xml_bytes, width, height):\n    objects, ignored = [], collections.Counter()\n    for obj in ET.fromstring(xml_bytes).findall(\'object\'):\n        name = obj.findtext(\'name\', \'\')\n        try:\n            source_type = shlex.split(name)[0].lower()\n        except (ValueError, IndexError):\n            source_type = name.lower()\n        code = W_MAP.get(source_type)\n        if code is None:\n            ignored[source_type] += 1\n            continue\n        box = obj.find(\'bndbox\')\n        # Pascal VOC inclusive, 1-based -> half-open 0-based pixel coordinates.\n        x1=max(0, int(float(box.findtext(\'xmin\')))-1)\n        y1=max(0, int(float(box.findtext(\'ymin\')))-1)\n        x2=min(width, int(float(box.findtext(\'xmax\'))))\n        y2=min(height, int(float(box.findtext(\'ymax\'))))\n        if x2 <= x1 or y2 <= y1:\n            raise ValueError(f\'Invalid XML box for {name}\')\n        crop = np.ones((y2-y1, x2-x1), np.uint8)\n        objects.append((CLASSES.index(code)+1, x1,y1,x2-x1,y2-y1,crop,\n                        (x2-x1)*(y2-y1)))\n    return objects, ignored\n\n\ndef group_key(domain, stem):\n    if domain == \'Graph-F\':\n        return re.match(r\'pcb_(\\d+)\', stem).group(1)\n    # Conservative grouping of front/back/top/bottom views from the same board.\n    return re.sub(r\'[_-](top|bottom).*$\', \'\', stem, flags=re.I)\n\n\ndef inventory(z):\n    records = []\n    for domain, expected in [(\'Graph-F\',(115,47)), (\'Graph-W\',(37,10))]:\n        for split, count in zip((\'train\',\'test\'), expected):\n            graphs = sorted(n for n in z.namelist()\n                            if f\'/{domain}/graphs/{split}/\' in n and n.endswith(\'.pt\'))\n            if len(graphs) != count:\n                raise ValueError(f\'{domain}/{split}: expected {count}, got {len(graphs)}\')\n            for graph in graphs:\n                stem = re.sub(r\'^graph_\\d+_\', \'\', Path(graph).stem)\n                prefix = graph.split(\'/graphs/\')[0]\n                image = prefix+\'/image_dir/\'+stem+(\'_image.png\' if domain==\'Graph-F\' else \'.jpg\')\n                ann = prefix+\'/mask_dir/\'+stem+(\'_mask.png\' if domain==\'Graph-F\' else \'.xml\')\n                if image not in z.namelist() or ann not in z.namelist():\n                    raise FileNotFoundError((image,ann))\n                records.append(dict(domain=domain, official_split=split, stem=stem,\n                                    group=domain+\':\'+group_key(domain,stem),\n                                    zip_image=image, zip_annotation=ann))\n    return records\n\n\ndef training_class_presence(z, records):\n    """Scan official training labels only; reserve rare classes for training."""\n    presence=collections.defaultdict(set)\n    color_to_code={tuple(color):code for color,code in zip(SOURCE_RGB,CLASSES)}\n    for i,r in enumerate([r for r in records if r[\'official_split\']==\'train\'],1):\n        if r[\'domain\']==\'Graph-F\':\n            im=Image.open(io.BytesIO(z.read(r[\'zip_annotation\']))).convert(\'RGB\')\n            colors=im.getcolors(maxcolors=4096)\n            if colors is None:\n                raise ValueError(\'Unexpected large color palette: \'+r[\'stem\'])\n            codes=set()\n            for _,color in colors:\n                if color==(0,0,0): continue\n                if color not in color_to_code:\n                    raise ValueError(f\'Unknown source mask color {color} in {r["stem"]}\')\n                codes.add(color_to_code[color])\n        else:\n            codes=set()\n            for obj in ET.fromstring(z.read(r[\'zip_annotation\'])).findall(\'object\'):\n                try: source=shlex.split(obj.findtext(\'name\',\'\'))[0].lower()\n                except (ValueError,IndexError): continue\n                if source in W_MAP: codes.add(W_MAP[source])\n        r[\'training_source_class_codes\']=sorted(codes)\n        presence[r[\'group\']].update(codes)\n        if i%25==0: print(f\'Scanned class support in {i}/152 official training boards\',flush=True)\n    return presence\n\n\ndef make_split(records, seed=42, val_fraction=.2, class_presence=None):\n    rng = np.random.default_rng(seed)\n    val_groups = set()\n    for domain in (\'Graph-F\',\'Graph-W\'):\n        groups = sorted({r[\'group\'] for r in records\n                         if r[\'domain\']==domain and r[\'official_split\']==\'train\'})\n        count=max(1,round(len(groups)*val_fraction))\n        if class_presence is None:\n            selected=rng.choice(groups,count,replace=False).tolist()\n        else:\n            # A group containing the last remaining training example of a class\n            # cannot be assigned to validation. No official test labels are read.\n            remaining=collections.Counter(c for g in groups for c in class_presence[g])\n            selected=[]\n            for g in rng.permutation(groups).tolist():\n                if all(remaining[c]>1 for c in class_presence[g]):\n                    selected.append(g)\n                    remaining.subtract(class_presence[g])\n                if len(selected)==count: break\n            if not selected:\n                raise ValueError(\'Cannot form validation while retaining training class support\')\n        val_groups.update(selected)\n    for r in records:\n        r[\'split\'] = (\'test\' if r[\'official_split\']==\'test\'\n                      else \'val\' if r[\'group\'] in val_groups else \'train\')\n    return records\n\n\ndef coco_base():\n    return dict(info={\'description\':\'GraphPCB; see data_mapping.json for limitations\'},\n                licenses=[], images=[], annotations=[],\n                categories=[dict(id=i+1,name=c) for i,c in enumerate(CLASSES)])\n\n\ndef annotation(objects, image_id, height, width, domain):\n    out=[]\n    for cid,x,y,w,h,crop,area in objects:\n        ann=dict(image_id=image_id,category_id=cid,bbox=[x,y,w,h],area=area,iscrowd=0)\n        # W placeholders are only needed by the shared MMDetection loader.\n        ann[\'segmentation\']=region_rle(crop,x,y,height,width)\n        ann[\'mask_valid\'] = domain==\'Graph-F\'\n        out.append(ann)\n    return out\n\n\ndef prepare(zip_path, out, tile=1024, overlap=256, seed=42, limit=None):\n    out=Path(out)\n    if not 0 <= overlap < tile:\n        raise ValueError(\'Require 0 <= overlap < tile\')\n    if (out/\'manifest.json\').exists():\n        raise FileExistsError(f\'{out}/manifest.json already exists; use a new output directory\')\n    z=open_source(zip_path)\n    signature=hashlib.sha256(json.dumps([(i.filename,i.CRC,i.file_size)\n                            for i in z.infolist()]).encode()).hexdigest()\n    records=inventory(z)\n    presence=training_class_presence(z,records)\n    records=make_split(records,seed,class_presence=presence)\n    counts=collections.Counter((r[\'domain\'],r[\'split\']) for r in records)\n    print(\'Board counts:\', dict(counts), flush=True)\n    full={s:coco_base() for s in (\'train\',\'val\',\'test\')}\n    tiles={s:coco_base() for s in (\'train\',\'val\')}\n    ignored=collections.Counter()\n    seen_hashes={}\n    selected=records if limit is None else records[:limit]\n    for image_id,r in enumerate(selected,1):\n        raw=z.read(r[\'zip_image\'])\n        sha=hashlib.sha256(raw).hexdigest()\n        if sha in seen_hashes and seen_hashes[sha][\'official_split\'] != r[\'official_split\']:\n            raise ValueError(\'Exact image duplicate across official train/test partitions\')\n        seen_hashes[sha]=r\n        image=np.array(Image.open(io.BytesIO(raw)).convert(\'RGB\'))\n        h,w=image.shape[:2]\n        image_rel=Path(\'boards\')/r[\'domain\']/Path(r[\'zip_image\']).name\n        (out/image_rel).parent.mkdir(parents=True,exist_ok=True)\n        (out/image_rel).write_bytes(raw)\n        if r[\'domain\']==\'Graph-F\':\n            mask=np.array(Image.open(io.BytesIO(z.read(r[\'zip_annotation\']))).convert(\'RGB\'))\n            if mask.shape != image.shape:\n                raise ValueError(\'Image/mask shape mismatch\')\n            objects=f_instances(mask)\n        else:\n            objects,ign=w_instances(z.read(r[\'zip_annotation\']),w,h)\n            if r[\'official_split\']==\'train\':\n                ignored.update(ign)\n        r.update(id=image_id,file_name=image_rel.as_posix(),width=w,height=h,\n                 sha256=sha,mask_valid=r[\'domain\']==\'Graph-F\')\n        split=r[\'split\']\n        full[split][\'images\'].append({k:r[k] for k in\n                                     (\'id\',\'file_name\',\'width\',\'height\',\'domain\',\'mask_valid\')})\n        full[split][\'annotations\'].extend(annotation(objects,image_id,h,w,r[\'domain\']))\n        if split!=\'test\':\n            for y in positions(h,tile,overlap):\n                for x in positions(w,tile,overlap):\n                    th,tw=min(tile,h-y),min(tile,w-x)\n                    clipped=[]\n                    for cid,ox,oy,ow,oh,crop,area in objects:\n                        xa,ya=max(x,ox),max(y,oy)\n                        xb,yb=min(x+tw,ox+ow),min(y+th,oy+oh)\n                        if xb<=xa or yb<=ya:\n                            continue\n                        part=crop[ya-oy:yb-oy,xa-ox:xb-ox]\n                        if not part.any():\n                            continue\n                        # Tighten the box after clipping; preserve every nonzero pixel.\n                        yy,xx=np.nonzero(part)\n                        ax,ay,bx,by=int(xx.min()),int(yy.min()),int(xx.max()+1),int(yy.max()+1)\n                        clipped.append((cid,xa-x+ax,ya-y+ay,bx-ax,by-ay,\n                                        part[ay:by,ax:bx],int(part.sum())))\n                    tid=len(tiles[split][\'images\'])+1\n                    rel=Path(\'tiles\')/split/f"{r[\'domain\']}__{r[\'stem\']}__{x}_{y}.jpg"\n                    (out/rel).parent.mkdir(parents=True,exist_ok=True)\n                    Image.fromarray(image[y:y+th,x:x+tw]).save(out/rel,quality=95)\n                    tiles[split][\'images\'].append(dict(id=tid,file_name=rel.as_posix(),\n                        width=tw,height=th,domain=r[\'domain\'],mask_valid=r[\'mask_valid\'],\n                        board_id=image_id))\n                    tiles[split][\'annotations\'].extend(annotation(clipped,tid,th,tw,r[\'domain\']))\n        print(f"[{image_id}/{len(selected)}] {r[\'domain\']} {r[\'stem\']}: {len(objects)} regions",flush=True)\n    for kind, sets in [(\'boards\',full),(\'tiles\',tiles)]:\n        for split,co in sets.items():\n            for aid,a in enumerate(co[\'annotations\'],1):\n                a[\'id\']=aid\n            dump(out/\'annotations\'/f\'{split}_{kind}.json\',co)\n            if kind==\'tiles\' and split==\'val\':\n                ids={im[\'id\'] for im in co[\'images\'] if im[\'mask_valid\']}\n                fco=dict(co,images=[im for im in co[\'images\'] if im[\'id\'] in ids],\n                         annotations=[a for a in co[\'annotations\'] if a[\'image_id\'] in ids])\n                dump(out/\'annotations\'/\'val_tiles_f.json\',fco)\n    mapping=dict(canonical_classes=[dict(id=i+1,code=c,source_rgb=SOURCE_RGB[i],\n                                        display_hex=DISPLAY_HEX[i]) for i,c in enumerate(CLASSES)],\n        source_palette=PALETTE_SOURCE,graph_w_type_mapping=W_MAP,\n        graph_w_ignored_train_types=dict(ignored),\n        graph_f_instance_rule=\'8-connected regions per exact RGB class; touching same-class objects remain merged; disconnected fragments remain separate\',\n        graph_w_mask_rule=\'Rectangles are loader placeholders. Mask loss is disabled for Graph-W. No Graph-W mask evaluation is valid.\',\n        graph_w_ambiguities=\'ic -> IC, switch -> SW. U versus IC and S versus SW remain distinct; ambiguous capacitor jumper, clock, pins, text, unknown etc are ignored.\',\n        input_images=\'Stored pixel values retained. No inferred HLS/RGB reversal.\',\n        external_pcb_images=0)\n    dump(out/\'data_mapping.json\',mapping)\n    support={}\n    for domain in (\'Graph-F\',\'Graph-W\'):\n        support[domain]={split:sorted({c for r in records if r[\'domain\']==domain and r[\'split\']==split\n                           for c in r.get(\'training_source_class_codes\',[])}) for split in (\'train\',\'val\')}\n    manifest=dict(seed=seed,tile=tile,overlap=overlap,records=selected,archive_signature=signature,\n                  training_class_support=support,\n                  official_train_boards=152,official_test_boards=57,\n                  distinct_training_boards=sum(r[\'split\']==\'train\' for r in selected),\n                  data_ratio=sum(r[\'split\']==\'train\' for r in selected)/152,\n                  debug_subset=limit is not None,\n                  limitation=\'Graph-W lacks pixel masks. Graph-F instance identities are recovered, not provided.\')\n    dump(out/\'manifest.json\',manifest)\n    return manifest\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--source\',\'--zip\',dest=\'source\',required=True,\n                   help=\'ZIP file or directory directly containing Graph-F and Graph-W\')\n    p.add_argument(\'--out\',required=True)\n    p.add_argument(\'--tile\',type=int,default=1024)\n    p.add_argument(\'--overlap\',type=int,default=256)\n    p.add_argument(\'--seed\',type=int,default=42)\n    p.add_argument(\'--limit\',type=int,help=\'Debug only; never use for final results\')\n    a=p.parse_args()\n    prepare(a.source,a.out,a.tile,a.overlap,a.seed,a.limit)\n', 'src/classification/train.py': '"""Fast cached baseline + short partial fine-tuning; test selection is forbidden."""\nimport argparse,sys,json,time,copy,random,shutil\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset,DataLoader,TensorDataset\nfrom PIL import Image\nfrom inference_classification import build,preprocessing,metrics,curves,save,boards\nTRIALS=[(\'small_cached160\',\'small\',160,0,20),(\'small_tail192\',\'small\',192,3,8),(\'b0_tail192\',\'b0\',192,2,8)]\nclass Crops(Dataset):\n    def __init__(self,root,rows,size,aug=False):self.root=root;self.rows=rows;self.tf=preprocessing(size,aug)\n    def __len__(self):return len(self.rows)\n    def __getitem__(self,i):\n        r=self.rows[i];return self.tf(Image.open(self.root/r[\'file\']).convert(\'RGB\')),r[\'label\']\ndef loader(ds,batch,shuffle=False,workers=2):return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=workers,pin_memory=torch.cuda.is_available())\ndef predict(m,dl,device):\n    m.eval();ys=[];ps=[];loss=0;n=0\n    with torch.inference_mode():\n        for x,y in dl:\n            logits=m(x.to(device));loss+=float(nn.functional.cross_entropy(logits,y.to(device),reduction=\'sum\'));n+=len(y);ys.extend(y.tolist());ps.append(logits.softmax(1).cpu().numpy())\n    return np.array(ys),np.concatenate(ps),loss/max(1,n)\ndef features(m,dl,device):\n    m.eval();xs=[];ys=[]\n    with torch.inference_mode():\n        for x,y in dl:xs.append(m.avgpool(m.features(x.to(device))).flatten(1).cpu());ys.append(y)\n    return TensorDataset(torch.cat(xs),torch.cat(ys))\ndef plots(dest,history,curve,y,p,t):\n    import matplotlib;matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    dest.mkdir(parents=True,exist_ok=True)\n    fig,ax=plt.subplots(1,2,figsize=(10,4))\n    for key in [\'P_macro\',\'R_macro\',\'F1_macro\']:ax[0].plot([r[\'threshold\'] for r in curve],[r[key] for r in curve],label=key)\n    ax[0].axvline(t,color=\'k\',ls=\'--\');ax[0].legend();ax[0].set_xlabel(\'Confidence threshold\')\n    for key in [\'train_loss\',\'val_loss\']:ax[1].plot([r[\'epoch\'] for r in history],[r[key] for r in history],label=key)\n    ax[1].legend();fig.tight_layout();fig.savefig(dest/\'curves_losses.png\');plt.close(fig)\n    pred=p.argmax(1);pred[p.max(1)<t]=25;cm=np.zeros((25,26),int);np.add.at(cm,(y,pred),1)\n    fig,ax=plt.subplots(figsize=(11,8));im=ax.imshow(cm,cmap=\'Blues\');ax.set(xlabel=\'Predicted (last column = rejected)\',ylabel=\'True class\',xticks=range(26),xticklabels=list(range(1,26))+[\'reject\'],yticks=range(25),yticklabels=range(1,26));fig.colorbar(im);fig.tight_layout();fig.savefig(dest/\'confusion_matrix.png\');plt.close(fig)\n    fig,ax=plt.subplots();ax.plot([r[\'epoch\'] for r in history],[r[\'val_accuracy\'] for r in history]);ax.set(xlabel=\'Epoch\',ylabel=\'Validation accuracy at threshold 0\');fig.savefig(dest/\'accuracy.png\');plt.close(fig)\ndef run(a):\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    if device==\'cpu\' and not a.allow_cpu:raise RuntimeError(\'Enable a Kaggle GPU\')\n    torch.set_num_threads(4);root=Path(a.data);out=Path(a.out);out.mkdir(parents=True,exist_ok=True)\n    rows=json.loads((root/\'crops.json\').read_text());parts={s:[r for r in rows if r[\'split\']==s] for s in [\'train\',\'val\',\'test\']}\n    statpath=out/\'classification_stats.json\';stats=json.loads(statpath.read_text()) if statpath.exists() else []\n    for name,arch,size,tail,epochs in TRIALS:\n        if any(r[\'config_name\']==name for r in stats):continue\n        random.seed(42);np.random.seed(42);torch.manual_seed(42)\n        start=time.perf_counter();m=build(arch,tail,True).to(device)\n        total=sum(p.numel() for p in m.parameters());trainable=sum(p.numel() for p in m.parameters() if p.requires_grad)\n        train_ds=Crops(root,parts[\'train\'],size,tail>0);val_ds=Crops(root,parts[\'val\'],size)\n        full_val=loader(val_ds,a.batch,workers=a.workers)\n        if tail==0:\n            train_ds=features(m,loader(train_ds,a.batch,workers=a.workers),device);val_ds=features(m,full_val,device);active=m.classifier\n        else:active=m\n        active=active.to(device);opt=torch.optim.AdamW([p for p in active.parameters() if p.requires_grad],lr=.001 if tail==0 else .0002,weight_decay=.0001)\n        counts=np.bincount([r[\'label\'] for r in parts[\'train\']],minlength=25);weight=np.sqrt(counts.max()/np.maximum(counts,1));weight=np.minimum(weight,5);weight=weight/weight.mean();criterion=nn.CrossEntropyLoss(weight=torch.tensor(weight,device=device,dtype=torch.float32))\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=device==\'cuda\');history=[];best=-1;stale=0;epoch0=0\n        work=out/name;work.mkdir(parents=True,exist_ok=True);last=work/\'last.pt\';bestpath=work/\'best_classification.ckpt\'\n        if last.exists():\n            c=torch.load(last,map_location=device,weights_only=False);m.load_state_dict(c[\'model\']);opt.load_state_dict(c[\'optimizer\']);scaler.load_state_dict(c[\'scaler\']);history=c[\'history\'];best=c[\'best\'];stale=c[\'stale\'];epoch0=c[\'epoch\']\n        for epoch in range(epoch0,epochs):\n            if stale>=3:break\n            active.train()\n            if tail:\n                for block in list(m.features.children())[:-tail]:block.eval()\n            running=0;n=0\n            for x,y in loader(train_ds,a.batch,True,a.workers):\n                x=x.to(device);y=y.to(device);opt.zero_grad(set_to_none=True)\n                with torch.autocast(device,enabled=device==\'cuda\'):loss=criterion(active(x),y)\n                scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(active.parameters(),5);scaler.step(opt);scaler.update();running+=float(loss)*len(y);n+=len(y)\n            y,p,vl=predict(active,loader(val_ds,a.batch,workers=a.workers),device);curve=curves(y,p);chosen=max(curve,key=lambda r:(r[\'S_cls\'],r[\'threshold\']))\n            history.append(dict(epoch=epoch+1,train_loss=running/n,val_loss=vl,val_accuracy=metrics(y,p,0)[\'accuracy\'],val_S_cls=chosen[\'S_cls\']))\n            if chosen[\'S_cls\']>best:\n                best=chosen[\'S_cls\'];stale=0\n                torch.save(dict(model=m.state_dict(),arch=arch,tail=tail,size=size,threshold=chosen[\'threshold\'],epoch=epoch+1,classes=list(range(25))),bestpath)\n            else:stale+=1\n            temp=last.with_suffix(\'.tmp\');torch.save(dict(model=m.state_dict(),optimizer=opt.state_dict(),scaler=scaler.state_dict(),history=history,best=best,stale=stale,epoch=epoch+1),temp);temp.replace(last)\n            print(name,history[-1],flush=True)\n        c=torch.load(bestpath,map_location=device,weights_only=False);m.load_state_dict(c[\'model\']);y,p,vl=predict(m,full_val,device);curve=curves(y,p);val=metrics(y,p,c[\'threshold\'])\n        save(work/\'history.json\',history);save(work/\'validation_curves.json\',curve);plots(work/\'validation\',history,curve,y,p,c[\'threshold\'])\n        stats.append(dict(config_name=name,seed=42,epochs=len(history),total_params=total,trainable_params=trainable,train_loss=history[c[\'epoch\']-1][\'train_loss\'],val_loss=vl,val_metrics=val,test_metrics=None,threshold=c[\'threshold\'],data_ratio=json.loads((root/\'manifest.json\').read_text())[\'data_ratio\'],runtime_seconds=time.perf_counter()-start,checkpoint_name=str(bestpath)))\n        save(statpath,stats)\n        del m,active,opt,train_ds,val_ds\n        if device==\'cuda\':torch.cuda.empty_cache()\n    winner=max(stats,key=lambda r:(r[\'val_metrics\'][\'S_cls\'],-r[\'trainable_params\']));save(out/\'selection_frozen.json\',dict(best_trial=winner[\'config_name\'],test_used_for_selection=False,trials=[dict(name=r[\'config_name\'],threshold=r[\'threshold\']) for r in stats]));shutil.copy2(winner[\'checkpoint_name\'],out/\'best_classification.ckpt\')\n    # Only now read/evaluate the entire prescribed test crops.\n    for r in stats:\n        if r[\'test_metrics\'] is not None:continue\n        c=torch.load(r[\'checkpoint_name\'],map_location=device,weights_only=False);m=build(c[\'arch\'],c[\'tail\']).to(device);m.load_state_dict(c[\'model\']);y,p,_=predict(m,loader(Crops(root,parts[\'test\'],c[\'size\']),a.batch,workers=a.workers),device)\n        r[\'test_metrics\']=metrics(y,p,r[\'threshold\']);work=out/r[\'config_name\'];curve=curves(y,p);save(work/\'test_curves.json\',curve);plots(work/\'test\',json.loads((work/\'history.json\').read_text()),curve,y,p,r[\'threshold\']);boards(m,c,root/\'test_boards.json\',root,work/\'test/overlays\',device);save(statpath,stats)\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',required=True);p.add_argument(\'--out\',required=True);p.add_argument(\'--batch\',type=int,default=128);p.add_argument(\'--workers\',type=int,default=2);p.add_argument(\'--allow-cpu\',action=\'store_true\');run(p.parse_args())\n'}
for rel,text in SOURCES.items():
 path=PROJECT/rel;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text,encoding='utf8')
def run(file,*args): subprocess.run([sys.executable,str(PROJECT/file),*map(str,args)],check=True,cwd=PROJECT)

## Prepare once
SOURCE may be an expanded GraphPCB directory or ZIP. If unique, the expanded folder is detected automatically. Crop extraction and the one-time feature cache can take several minutes.

In [ ]:
if not SOURCE:
 roots=[p.parent for p in Path('/kaggle/input').rglob('Graph-F') if (p/'image_dir').is_dir() and (p.parent/'Graph-W/graphs').is_dir()]
 assert len(roots)==1,'Set SOURCE explicitly to your GraphPCB folder or ZIP'
 SOURCE=str(roots[0])
if not (DATA/'manifest.json').exists(): run('src/classification/prepare.py','--source',SOURCE,'--out',DATA)
OUT.mkdir(parents=True,exist_ok=True)
shutil.copy2(DATA/'manifest.json',OUT/'manifest.json')

## Run three short trials
The cached head baseline is cheapest. Partial fine-tuning is the accuracy challenger. All trials save an epoch checkpoint and can resume. Frozen selections precede the complete test evaluation. Do not retune using test results.

In [ ]:
run('src/classification/train.py','--data',DATA,'--out',OUT,'--batch',128,'--workers',2)
(PROJECT/'checkpoints').mkdir(exist_ok=True)
shutil.copy2(OUT/'best_classification.ckpt',PROJECT/'checkpoints/best_classification.ckpt')

In [ ]:
import pandas as pd
from IPython.display import display
stats=json.loads((OUT/'classification_stats.json').read_text())
display(pd.DataFrame([dict(trial=r['config_name'],total_params=r['total_params'],trainable_params=r['trainable_params'],epochs=r['epochs'],val_S=r['val_metrics']['S_cls'],test_F1=r['test_metrics']['F1_macro'],test_S=r['test_metrics']['S_cls']) for r in stats]))
print((OUT/'selection_frozen.json').read_text())

## Save outputs
Save the Kaggle version with outputs to retain optimizer checkpoints for resume. The portable ZIP below excludes last.pt and all dataset images. It includes source, metrics, plots, test overlays and inference checkpoints. This is the classification portion, to combine with your segmentation submission.

In [ ]:
import zipfile
package=Path('/kaggle/working/PCB_Classification_results.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
 for root,prefix in [(PROJECT,Path('Classification')),(OUT,Path('Classification/results/classification'))]:
  for f in root.rglob('*'):
   if f.is_file() and '__pycache__' not in f.parts and f.name!='last.pt':z.write(f,prefix/f.relative_to(root))
print(package)